# Arabic character recognition with a small Vision Transformer

This model predicts one of the 38 character labels from a single image. It does
not read connected words or generate text. A Vision Transformer divides an image
into patches, embeds them as tokens, and uses attention to combine information.

Run the cells in order using `D:/python/.venv/Scripts/python.exe` as your kernel.
Images are loaded in batches rather than storing the whole dataset in RAM.
Start with 10 epochs and compare validation results with your CNN. A transformer
is an alternative architecture; it is not guaranteed to outperform the CNN.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

keras.utils.set_random_seed(42)
DATA_DIR = Path(r"D:\python\CNNs\arabic letters\Dataset")
OUTPUT_DIR = DATA_DIR.parent.parent
IMAGE_SIZE = 64
PATCH_SIZE = 8
EMBED_DIM = 64
BATCH_SIZE = 64
EPOCHS = 10
print("TensorFlow:", tf.__version__)

## 1. Find images and split their paths

Folder names are the class labels. Only PNG files are included. The split is
70% training, 15% validation, and 15% testing, stratified by class. Save the label
order with the model because output index 0 refers to the first name in this list.

This is a split by image. If images share a writer or are augmented versions of
the same original, use those source groups for splitting instead. Otherwise the
test score can overestimate performance on new handwriting. Writer/source IDs
are not supplied by this folder-based loader.

In [ ]:
class_names = sorted(folder.name for folder in DATA_DIR.iterdir() if folder.is_dir())
paths, labels = [], []
for label, name in enumerate(class_names):
    for path in sorted((DATA_DIR / name).glob("*.png")):
        paths.append(str(path))
        labels.append(label)

paths = np.asarray(paths)
labels = np.asarray(labels, dtype=np.int32)
train_paths, remaining_paths, train_labels, remaining_labels = train_test_split(
    paths, labels, test_size=0.30, random_state=42, stratify=labels,
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    remaining_paths, remaining_labels, test_size=0.50,
    random_state=42, stratify=remaining_labels,
)
assert set(train_paths).isdisjoint(val_paths)
assert set(train_paths).isdisjoint(test_paths)
assert set(val_paths).isdisjoint(test_paths)
print("Classes:", len(class_names), class_names)
print("Images:", len(paths))
print("Train / validation / test:", len(train_paths), len(val_paths), len(test_paths))

In [ ]:
def load_image(path, label):
    image = tf.io.decode_png(tf.io.read_file(path), channels=1)
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image / 255.0, label


def make_dataset(image_paths, image_labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, image_labels))
    if training:
        dataset = dataset.shuffle(len(image_paths), seed=42)
    return (dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

In [ ]:
sample_images, sample_labels = next(iter(train_ds))
plt.figure(figsize=(9, 5))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(sample_images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    plt.title(class_names[int(sample_labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Build the transformer

Each 64 by 64 image becomes 64 non-overlapping patches of 8 by 8 pixels. A dense
layer maps each patch to 64 numbers. Learned position embeddings tell the model
where each patch came from. Two attention blocks mix information between patches.
Average pooling combines the tokens before the final character classifier.

Small rotations and translations augment only training images. Horizontal and
vertical flips are omitted because they can change character identity.
The registered custom layer below allows the saved model to be loaded after
running its definition.

In [ ]:
@keras.utils.register_keras_serializable(package="ArabicViT")
class PatchEmbedding(layers.Layer):
    def __init__(self, image_size, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = layers.Dense(embed_dim)
        self.position_embedding = layers.Embedding(self.num_patches, embed_dim)

    def call(self, images):
        patches = tf.image.extract_patches(
            images, sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1], padding="VALID",
        )
        patches = tf.reshape(patches, (
            tf.shape(images)[0], self.num_patches, self.patch_size ** 2,
        ))
        positions = tf.range(self.num_patches)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        return dict(super().get_config(), image_size=self.image_size,
                    patch_size=self.patch_size, embed_dim=self.embed_dim)


inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))
x = layers.RandomRotation(0.03, fill_mode="nearest")(inputs)
x = layers.RandomTranslation(0.05, 0.05, fill_mode="nearest")(x)
x = PatchEmbedding(IMAGE_SIZE, PATCH_SIZE, EMBED_DIM)(x)

for _ in range(2):
    normalized = layers.LayerNormalization(epsilon=1e-6)(x)
    attention = layers.MultiHeadAttention(
        num_heads=4, key_dim=EMBED_DIM // 4, dropout=0.1,
    )(normalized, normalized)
    x = layers.Add()([x, attention])
    normalized = layers.LayerNormalization(epsilon=1e-6)(x)
    mlp = layers.Dense(EMBED_DIM * 2, activation="gelu")(normalized)
    mlp = layers.Dropout(0.1)(mlp)
    mlp = layers.Dense(EMBED_DIM)(mlp)
    x = layers.Add()([x, mlp])

x = layers.LayerNormalization(epsilon=1e-6)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs, name="small_arabic_vit")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"],
)
model.summary()

## 3. Train

Early stopping restores the weights with the lowest validation loss. Learning
rate reduction can help if validation loss stops improving. Training on all
94,269 images may take time on a CPU. Test data is kept out of training and model
selection.

In [ ]:
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5,
        ),
    ],
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history.history[metric], label="Training")
    ax.plot(history.history["val_" + metric], label="Validation")
    ax.set(xlabel="Epoch (starting at 0)", ylabel=metric)
    ax.legend()
plt.tight_layout()
plt.show()

## 4. Evaluate after choosing the model settings

Use the untouched test set once you finish choosing settings with validation
data. The class report shows which letters or digits need attention. A confusion
matrix counts the true/predicted pairs; row and column indices follow class_names.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy:.2%}")
probabilities = model.predict(test_ds)
predictions = probabilities.argmax(axis=1)
print(classification_report(
    test_labels, predictions, labels=np.arange(len(class_names)),
    target_names=class_names, zero_division=0,
))
matrix = confusion_matrix(test_labels, predictions, labels=np.arange(len(class_names)))
plt.figure(figsize=(10, 8))
plt.imshow(matrix, cmap="Blues")
plt.xlabel("Predicted class index")
plt.ylabel("True class index")
plt.colorbar(label="Number of images")
plt.show()

In [ ]:
# Prefer mistakes to inspect what confuses the model.
wrong = np.flatnonzero(predictions != test_labels)
pool = wrong if len(wrong) else np.arange(len(test_labels))
indices = np.random.default_rng(42).choice(pool, size=min(16, len(pool)), replace=False)
plt.figure(figsize=(11, 9))
for i, index in enumerate(indices):
    image, _ = load_image(test_paths[index], test_labels[index])
    plt.subplot(4, 4, i + 1)
    plt.imshow(image[:, :, 0], cmap="gray", vmin=0, vmax=1)
    plt.title(f"True: {class_names[test_labels[index]]} | Pred: {class_names[predictions[index]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Save and use the model

For a new image, crop around one character and match the training images'
orientation and foreground/background colors. The top three values below are
softmax scores, not calibrated certainty estimates.

To load in a fresh kernel, run the imports and PatchEmbedding class definition
first, then use keras.models.load_model on arabic_vit.keras and read the saved
JSON label list. Do not rebuild the label order from a different directory.

In [ ]:
model.save(OUTPUT_DIR / "arabic_vit.keras")
(OUTPUT_DIR / "arabic_vit_classes.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("Saved to:", OUTPUT_DIR)


def predict_character(image_path):
    image, _ = load_image(str(image_path), 0)
    scores = model.predict(image[tf.newaxis], verbose=0)[0]
    best = scores.argsort()[-3:][::-1]
    return [(class_names[index], float(scores[index])) for index in best]


# Replace this path with your own single-character PNG when ready.
predict_character(test_paths[0])